# পাঠ ০৫ - এজেন্টিক RAG


## সেটআপ

এই নোটবুকটি Microsoft Agent Framework ব্যবহার করে Agentic RAG (Retrieval-Augmented Generation) প্যাটার্ন প্রদর্শন করে।

**প্রয়োজনীয়তা:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — আপনার Azure AI Search সার্ভিস এন্ডপয়েন্ট
- `AZURE_SEARCH_API_KEY` — আপনার Azure AI Search API কী
- পরিবেশ ভেরিয়েবল মাধ্যমে কনফিগার করা Azure OpenAI ডিপ্লয়মেন্ট
- Azure CLI অনুমোদিত (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Agentic RAG কী?

পारম্পরিক RAG একটি নির্দিষ্ট পাইপলাইন অনুসরণ করে: নথি পুনরুদ্ধার করুন, তারপরে একটি প্রতিক্রিয়া তৈরি করুন। **Agentic RAG** আরও এগিয়ে যায় এজেন্টকে স্বায়ত্তশাসন দেয় **কখন** এবং **কিভাবে** তথ্য পুনরুদ্ধার করতে হবে তা সিদ্ধান্ত নেওয়ার জন্য।

Agentic RAG-র সাহায্যে, এজেন্ট করতে পারে:
- একটি প্রশ্নের উত্তর দেওয়ার আগে পুনরুদ্ধার প্রয়োজন কিনা তা **সিদ্ধান্ত** নেওয়া
- কোন তথ্য উৎস বা সরঞ্জাম ব্যবহার করে অনুসন্ধান করতে হবে তা **পছন্দ করা**
- পুনরুদ্ধারকৃত ফলাফলগুলি **মূল্যায়ন** করা এবং প্রথম প্রচেষ্টা যথেষ্ট না হলে পুনরায় অনুসন্ধান চালানো
- একাধিক পুনরুদ্ধার ধাপ থেকে তথ্য **মিশ্রিত** করে একটি সঙ্গতিপূর্ণ উত্তর তৈরি করা

এটি এজেন্টকে একটি স্থির পুনরুদ্ধার-তারপর-উৎপন্ন পাইপলাইনের তুলনায় আরও নমনীয় এবং সঠিক করে তোলে।


## একটি অনুসন্ধান টুল তৈরি করা

Agentic RAG-এ, বাহ্যিক ডেটা উৎসগুলোকে **টুলস** হিসেবে মোড়ানো হয় যা এজেন্ট প্রয়োজনে আহ্বান করতে পারে। এটি এজেন্টকে পুনরুদ্ধারকে কেবল একটি অন্য কর্ম হিসেবে বিবেচনা করার সুযোগ দেয়, যা বাধ্যতামূলক ধাপ নয়।

নিচে আমরা একটি ভ্রমণ জ্ঞানের ভিত্তি সংজ্ঞায়িত করেছি এবং এটি এমন একটি টুল হিসেবে উন্মোচন করেছি যা এজেন্ট গন্তব্য তথ্য_lookup করার জন্য কল করতে পারে।


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG এজেন্ট তৈরি করা

এখন আমরা একটি এজেন্ট তৈরি করছি যাকে নির্দেশ দেওয়া হয়েছে **প্রতিবার উত্তর দেওয়ার আগে তথ্য অনুসন্ধান করতে হবে**। এজেন্টটি তার নিজেদের প্রশিক্ষণ ডেটার উপর নির্ভর করার পরিবর্তে জ্ঞানভাণ্ডারে ভিত্তি করে তার উত্তরগুলো তৈরি করতে `search_travel_knowledge` টুলটি ব্যবহার করে।


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## পুনরাবৃত্তিমূলক অনুসন্ধান — মেকার-চেকার প্যাটার্ন

Agentic RAG এর একটি মূল সুবিধা হল **পুনরাবৃত্তিমূলক অনুসন্ধান**। এজেন্ট প্রাথমিক ফলাফল যাচাই, পরিমার্জন বা সম্প্রসারণের জন্য একাধিক দফা অনুসন্ধান করতে পারে — যা একটি "মেকার-চেকার" ওয়ার্কফ্লোর মতো:

1. **মেকার ধাপ**: এজেন্ট প্রাথমিক তথ্য সংগ্রহ করে এবং একটি উত্তর খসড়া প্রস্তুত করে।
2. **চেকার ধাপ**: এজেন্ট বিশদ যাচাই বা ফাঁক পূরণের জন্য অতিরিক্ত অনুসন্ধান করে।

নীচে, এজেন্টকে এমন একটি প্রশ্ন করা হয়েছে যা একাধিক গন্তব্যের তুলনা প্রয়োজন, ফলে তাকে একাধিকবার অনুসন্ধান করতে উৎসাহিত করা হয়।


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## সারসংক্ষেপ

এই পাঠে আপনি শিখেছেন কিভাবে Microsoft Agent Framework ব্যবহার করে একটি **Agentic RAG** সিস্টেম তৈরি করবেন:

- **Agentic RAG** এজেন্টদের স্বতঃস্ফূর্তভাবে সিদ্ধান্ত নিতে দেয় কখন তথ্য আহরণ করতে হবে, ফলে তথ্য আহরণ স্থির না হয়ে গতিশীল হয়।
- **টুলসকে ডেটা উৎস হিসাবে**: বাহ্যিক জ্ঞানভান্ডার (যেমন Azure AI Search) টুল হিসেবে প্যাকেজ করা হয় যা এজেন্ট ব্যবহার করতে পারে।
- **পুনরাবৃত্ত তথ্য আহরণ**: মেকার-চেকার প্যাটার্ন এজেন্টকে একাধিক তথ্য আহরণের দফা সম্পাদন করতে সক্ষম করে — অনুসন্ধান, যাচাই, এবং পরিশীলন — তারপর চূড়ান্ত উত্তর প্রদান।

প্রোডাকশনে, বড় পরিমাণের ভ্রমণ ডকুমেন্ট আহরণের জন্য আপনি মেমোরিতে সংরক্ষিত `TRAVEL_KNOWLEDGE_BASE` এর পরিবর্তে একটি বাস্তব Azure AI Search ইনডেক্স ব্যবহার করবেন।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
